In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Gumawa ng transpiler plugin

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';





<details>
<summary><b>Mga bersyon ng package</b></summary>

Ang code sa pahinang ito ay ginawa gamit ang mga sumusunod na kinakailangan.
Inirerekomenda naming gamitin ang mga bersyong ito o mas bago.

```
qiskit[all]~=2.3.0
```
</details>
Ang paggawa ng [transpiler plugin](transpiler-plugins) ay isang magandang paraan para ibahagi ang iyong transpilation code sa mas malawak na komunidad ng Qiskit, na nagbibigay-daan sa ibang mga user na makinabang sa functionality na iyong ginawa. Salamat sa iyong interes na mag-ambag sa komunidad ng Qiskit!

Bago ka gumawa ng transpiler plugin, kailangan mong magpasya kung anong uri ng plugin ang angkop para sa iyong sitwasyon. May tatlong uri ng transpiler plugins:
- [**Transpiler stage plugin**](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins). Piliin ito kung nagtatakda ka ng pass manager na maaaring ipalit sa isa sa [6 na yugto](transpiler-stages) ng isang preset staged pass manager.
- [**Unitary synthesis plugin**](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin). Piliin ito kung ang iyong transpilation code ay tumatanggap ng unitary matrix bilang input (kinakatawan bilang Numpy array) at naglalabas ng paglalarawan ng quantum circuit na nagpapatupad ng unitary na iyon.
- [**High-level synthesis plugin**](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin). Piliin ito kung ang iyong transpilation code ay tumatanggap ng "high-level na object" tulad ng Clifford operator o linear function bilang input at naglalabas ng paglalarawan ng quantum circuit na nagpapatupad ng high-level na object na iyon. Ang mga high-level na object ay kinakatawan ng mga subclass ng [Operation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Operation) class.

Kapag natukoy mo na kung anong uri ng plugin ang gagawin, sundin ang mga hakbang na ito para gawin ang plugin:

1. Gumawa ng subclass ng naaangkop na abstract plugin class:
   - [PassManagerStagePlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePlugin) para sa transpiler stage plugin,
   - [UnitarySynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin) para sa unitary synthesis plugin, at
   - [HighLevelSynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin) para sa high-level synthesis plugin.
2. I-expose ang class bilang [setuptools entry point](https://setuptools.pypa.io/en/latest/userguide/entry_point.html) sa metadata ng package, karaniwan sa pamamagitan ng pag-edit ng `pyproject.toml`, `setup.cfg`, o `setup.py` na file ng iyong Python package.

Walang limitasyon sa bilang ng mga plugin na maaaring tukuyin ng isang package, ngunit ang bawat plugin ay dapat may natatanging pangalan. Kasama sa Qiskit SDK ang ilang built-in na plugins, at ang kanilang mga pangalan ay nakalaan din. Ang mga nakalaan na pangalan ay:

- Transpiler stage plugins: Tingnan ang [talahanayan na ito](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins#plugin-stages).
- Unitary synthesis plugins: `default`, `aqc`, `sk`
- High-level synthesis plugins:

| Klase ng Operation | Pangalan ng Operation | Mga nakalaan na pangalan |
| --- | --- | --- |
| [Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford#clifford) | `clifford` | `default`, `ag`, `bm`, `greedy`, `layers`, `lnn` |
| [LinearFunction](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction#linearfunction) | `linear_function` | `default`, `kms`, `pmh` |
| [PermutationGate](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.PermutationGate#permutationgate) | `permutation` | `default`, `kms`, `basic`, `acg`, `token_swapper` |

Sa mga susunod na seksyon, nagpapakita tayo ng mga halimbawa ng mga hakbang na ito para sa iba't ibang uri ng plugins. Sa mga halimbawang ito, ipinapalagay natin na gumagawa tayo ng Python package na tinatawag na `my_qiskit_plugin`. Para sa impormasyon tungkol sa paggawa ng Python packages, maaari mong tingnan ang [tutorial na ito](https://packaging.python.org/en/latest/tutorials/packaging-projects/) mula sa website ng Python.
## Halimbawa: Gumawa ng transpiler stage plugin
Sa halimbawang ito, gumagawa tayo ng transpiler stage plugin para sa `layout` na yugto (tingnan ang [Transpiler stages](transpiler-stages) para sa paglalarawan ng 6 na yugto ng built-in na transpilation pipeline ng Qiskit).
Ang aming plugin ay nagpapatakbo lang ng [VF2Layout](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.VF2Layout) para sa bilang ng mga trial na depende sa hiniling na antas ng optimization.

Una, gumagawa tayo ng subclass ng [PassManagerStagePlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePlugin). May isang method na kailangan nating i-implement, na tinatawag na [`pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePlugin#pass_manager). Tinatanggap ng method na ito bilang input ang isang [PassManagerConfig](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManagerConfig) at ibinabalik ang pass manager na ating tinutukoy. Ang PassManagerConfig na object ay nag-iimbak ng impormasyon tungkol sa target na backend, tulad ng coupling map at basis gates nito.

In [1]:
# This import is needed for python versions prior to 3.10
from __future__ import annotations

from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import VF2Layout
from qiskit.transpiler.passmanager_config import PassManagerConfig
from qiskit.transpiler.preset_passmanagers import common
from qiskit.transpiler.preset_passmanagers.plugin import (
    PassManagerStagePlugin,
)


class MyLayoutPlugin(PassManagerStagePlugin):
    def pass_manager(
        self,
        pass_manager_config: PassManagerConfig,
        optimization_level: int | None = None,
    ) -> PassManager:
        layout_pm = PassManager(
            [
                VF2Layout(
                    coupling_map=pass_manager_config.coupling_map,
                    properties=pass_manager_config.backend_properties,
                    max_trials=optimization_level * 10 + 1,
                    target=pass_manager_config.target,
                )
            ]
        )
        layout_pm += common.generate_embed_passmanager(
            pass_manager_config.coupling_map
        )
        return layout_pm

Ngayon, i-expose natin ang plugin sa pamamagitan ng pagdaragdag ng entry point sa metadata ng aming Python package.
Dito, ipinapalagay natin na ang class na aming tinukoy ay naka-expose sa isang module na tinatawag na `my_qiskit_plugin`, halimbawa sa pamamagitan ng pag-import nito sa `__init__.py` na file ng `my_qiskit_plugin` module.
Ie-edit natin ang `pyproject.toml`, `setup.cfg`, o `setup.py` na file ng aming package (depende kung aling uri ng file ang pinili mong gamitin para i-store ang metadata ng iyong Python project):

<Tabs>
  <TabItem value="package-table-toml" label="pyproject.toml" default>
    ```toml
    [project.entry-points."qiskit.transpiler.layout"]
    "my_layout" = "my_qiskit_plugin:MyLayoutPlugin"
    ```
  </TabItem>
  <TabItem value="package-table-cfg" label="setup.cfg">
    ```ini
    [options.entry_points]
    qiskit.transpiler.layout =
        my_layout = my_qiskit_plugin:MyLayoutPlugin
    ```
  </TabItem>
  <TabItem value="package-table-py" label="setup.py">
    ```python
    from setuptools import setup

    setup(
        # ...,
        entry_points={
            'qiskit.transpiler.layout': [
                'my_layout = my_qiskit_plugin:MyLayoutPlugin',
            ]
        }
    )
    ```
  </TabItem>
</Tabs>
Tingnan ang [talahanayan ng mga transpiler plugin stage](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins#stage-table) para sa mga entry point at mga inaasahan para sa bawat transpiler stage.

Para tingnan kung matagumpay na nadetect ng Qiskit ang iyong plugin, i-install ang iyong plugin package at sundin ang mga tagubilin sa [Transpiler plugins](transpiler-plugins#list-available-transpiler-stage-plugins) para sa pag-lista ng mga naka-install na plugins, at tiyakin na lumalabas ang iyong plugin sa listahan:

In [2]:
from qiskit.transpiler.preset_passmanagers.plugin import list_stage_plugins

list_stage_plugins("layout")

['default', 'dense', 'sabre', 'trivial']

If our example plugin were installed, then the name `my_layout` would appear in this list.

If you want to use a built-in transpiler stage as the starting point for your transpiler stage plugin, you can obtain the pass manager for a built-in transpiler stage using [PassManagerStagePluginManager](/docs/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePluginManager#passmanagerstagepluginmanager). The following code cell shows how to do this to obtain the built-in optimization stage for optimization level 3.

In [3]:
from qiskit.transpiler.preset_passmanagers.plugin import (
    PassManagerStagePluginManager,
)

# Initialize the plugin manager
plugin_manager = PassManagerStagePluginManager()

# Here we create a pass manager config to use as an example.
# Instead, you should use the pass manager config that you already received as input
# to the pass_manager method of your PassManagerStagePlugin.
pass_manager_config = PassManagerConfig()

# Obtain the desired built-in transpiler stage
optimization = plugin_manager.get_passmanager_stage(
    "optimization", "default", pass_manager_config, optimization_level=3
)

Kung naka-install ang aming halimbawang plugin, lilitaw ang pangalang `my_layout` sa listahang ito.

Kung gusto mong gamitin ang isang built-in na transpiler stage bilang panimulang punto para sa iyong transpiler stage plugin, maaari mong makuha ang pass manager para sa isang built-in na transpiler stage gamit ang [PassManagerStagePluginManager](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePluginManager#passmanagerstagepluginmanager). Ipinapakita ng sumusunod na code cell kung paano ito gawin para makuha ang built-in na optimization stage para sa optimization level 3.

In [4]:
import numpy as np
from qiskit.circuit import QuantumCircuit, QuantumRegister
from qiskit.converters import circuit_to_dag
from qiskit.dagcircuit.dagcircuit import DAGCircuit
from qiskit.quantum_info import Operator
from qiskit.transpiler.passes import UnitarySynthesis
from qiskit.transpiler.passes.synthesis.plugin import UnitarySynthesisPlugin


class MyUnitarySynthesisPlugin(UnitarySynthesisPlugin):
    @property
    def supports_basis_gates(self):
        # Returns True if the plugin can target a list of basis gates
        return True

    @property
    def supports_coupling_map(self):
        # Returns True if the plugin can synthesize for a given coupling map
        return False

    @property
    def supports_natural_direction(self):
        # Returns True if the plugin supports a toggle for considering
        # directionality of 2-qubit gates
        return False

    @property
    def supports_pulse_optimize(self):
        # Returns True if the plugin can optimize pulses during synthesis
        return False

    @property
    def supports_gate_lengths(self):
        # Returns True if the plugin can accept information about gate lengths
        return False

    @property
    def supports_gate_errors(self):
        # Returns True if the plugin can accept information about gate errors
        return False

    @property
    def supports_gate_lengths_by_qubit(self):
        # Returns True if the plugin can accept information about gate lengths
        # (The format of the input differs from supports_gate_lengths)
        return False

    @property
    def supports_gate_errors_by_qubit(self):
        # Returns True if the plugin can accept information about gate errors
        # (The format of the input differs from supports_gate_errors)
        return False

    @property
    def min_qubits(self):
        # Returns the minimum number of qubits the plugin supports
        return None

    @property
    def max_qubits(self):
        # Returns the maximum number of qubits the plugin supports
        return None

    @property
    def supported_bases(self):
        # Returns a dictionary of supported bases for synthesis
        return None

    def run(self, unitary: np.ndarray, **options) -> DAGCircuit:
        basis_gates = options["basis_gates"]
        synth_pass = UnitarySynthesis(basis_gates, min_qubits=3)
        qubits = QuantumRegister(3)
        circuit = QuantumCircuit(qubits)
        circuit.append(Operator(unitary).to_instruction(), qubits)
        dag_circuit = synth_pass.run(circuit_to_dag(circuit))
        return dag_circuit

## Halimbawa: Gumawa ng unitary synthesis plugin
Sa halimbawang ito, gagawa tayo ng unitary synthesis plugin na gumagamit lang ng built-in na [UnitarySynthesis](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.UnitarySynthesis#unitarysynthesis) transpilation pass para mag-synthesize ng gate. Syempre, ang iyong sariling plugin ay gagawa ng mas kawili-wiling bagay kaysa riyan.

Ang [UnitarySynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin) class ay tumutukoy sa interface at kontrata para sa unitary synthesis
plugins. Ang pangunahing method ay
[`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin#run),
na tumatanggap bilang input ng Numpy array na nag-iimbak ng unitary matrix
at nagbabalik ng [DAGCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.dagcircuit.DAGCircuit) na kumakatawan sa circuit na na-synthesize mula sa unitary matrix na iyon.
Bukod sa `run` method, may ilang property method na kailangang tukuyin.
Tingnan ang [UnitarySynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin) para sa dokumentasyon ng lahat ng kinakailangang property.

Gawin natin ang ating UnitarySynthesisPlugin subclass:

In [5]:
from qiskit.transpiler.passes.synthesis import unitary_synthesis_plugin_names

unitary_synthesis_plugin_names()

['aqc', 'clifford', 'default', 'gridsynth', 'sk']

Kung nalaman mong hindi sapat ang mga input na available sa [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin#run)
method para sa iyong layunin, mangyaring [mag-open ng issue](https://github.com/Qiskit/qiskit/issues/new/choose) na nagpapaliwanag ng iyong mga kinakailangan. Ang mga pagbabago sa plugin interface, tulad ng pagdaragdag ng karagdagang opsyonal na input, ay gagawin sa isang paraan na backward compatible para hindi ito mangailangan ng mga pagbabago mula sa mga kasalukuyang plugins.

> **Note:** Lahat ng method na may prefix na `supports_` ay nakalaan sa isang `UnitarySynthesisPlugin` derived class bilang bahagi ng interface. Hindi ka dapat mag-define ng anumang custom na `supports_*` method sa isang subclass na hindi tinukoy sa abstract class.

Ngayon, i-expose natin ang plugin sa pamamagitan ng pagdaragdag ng entry point sa metadata ng aming Python package.
Dito, ipinapalagay natin na ang class na aming tinukoy ay naka-expose sa isang module na tinatawag na `my_qiskit_plugin`, halimbawa sa pamamagitan ng pag-import nito sa `__init__.py` na file ng `my_qiskit_plugin` module.
Ie-edit natin ang `pyproject.toml`, `setup.cfg`, o `setup.py` na file ng aming package:

<Tabs>
  <TabItem value="package-table-toml" label="pyproject.toml" default>
    ```toml
    [project.entry-points."qiskit.unitary_synthesis"]
    "my_unitary_synthesis" = "my_qiskit_plugin:MyUnitarySynthesisPlugin"
    ```
  </TabItem>
  <TabItem value="package-table-cfg" label="setup.cfg">
    ```ini
    [options.entry_points]
    qiskit.unitary_synthesis =
        my_unitary_synthesis = my_qiskit_plugin:MyUnitarySynthesisPlugin
    ```
  </TabItem>
  <TabItem value="package-table-py" label="setup.py">
    ```python
    from setuptools import setup

    setup(
        # ...,
        entry_points={
            'qiskit.unitary_synthesis': [
                'my_unitary_synthesis = my_qiskit_plugin:MyUnitarySynthesisPlugin',
            ]
        }
    )
    ```
  </TabItem>
</Tabs>

Tulad ng dati, kung ang iyong project ay gumagamit ng `setup.cfg` o `setup.py` sa halip na `pyproject.toml`, tingnan ang [dokumentasyon ng setuptools](https://setuptools.pypa.io/en/latest/userguide/entry_point.html) para malaman kung paano i-adapt ang mga linyang ito para sa iyong sitwasyon.

Para tingnan kung matagumpay na nadetect ng Qiskit ang iyong plugin, i-install ang iyong plugin package at sundin ang mga tagubilin sa [Transpiler plugins](transpiler-plugins#list-available-unitary-synthesis-plugins) para sa pag-lista ng mga naka-install na plugins, at tiyakin na lumalabas ang iyong plugin sa listahan:

In [6]:
from qiskit.synthesis import synth_clifford_bm
from qiskit.transpiler.passes.synthesis.plugin import HighLevelSynthesisPlugin


class MyCliffordSynthesisPlugin(HighLevelSynthesisPlugin):
    def run(
        self,
        high_level_object,
        coupling_map=None,
        target=None,
        qubits=None,
        **options,
    ) -> QuantumCircuit:
        if high_level_object.num_qubits <= 3:
            return synth_clifford_bm(high_level_object)
        else:
            return None

This plugin synthesizes objects of type [Clifford](/docs/api/qiskit/qiskit.quantum_info.Clifford) that have
at most 3 qubits, using the `synth_clifford_bm` method.

Now, we expose the plugin by adding an entry point in our Python package metadata.
Here, we assume that the class we defined is exposed in a module called `my_qiskit_plugin`, for example by being imported in the `__init__.py` file of the `my_qiskit_plugin` module.
We edit the `pyproject.toml`, `setup.cfg`, or `setup.py` file of our package:

<Tabs>
  <TabItem value="package-table-toml" label="pyproject.toml" default>
    ```toml
    [project.entry-points."qiskit.synthesis"]
    "clifford.my_clifford_synthesis" = "my_qiskit_plugin:MyCliffordSynthesisPlugin"
    ```
  </TabItem>
  <TabItem value="package-table-cfg" label="setup.cfg">
    ```ini
    [options.entry_points]
    qiskit.synthesis =
        clifford.my_clifford_synthesis = my_qiskit_plugin:MyCliffordSynthesisPlugin
    ```
  </TabItem>
  <TabItem value="package-table-py" label="setup.py">
    ```python
    from setuptools import setup

    setup(
        # ...,
        entry_points={
            'qiskit.synthesis': [
                'clifford.my_clifford_synthesis = my_qiskit_plugin:MyCliffordSynthesisPlugin',
            ]
        }
    )
    ```
  </TabItem>
</Tabs>

The `name` consists of two parts separated by a dot (`.`):
- The name of the type of [Operation](/docs/api/qiskit/qiskit.circuit.Operation) that the plugin synthesizes (in this case, `clifford`). Note that this string corresponds to the [`name`](/docs/api/qiskit/qiskit.circuit.Operation#name) attribute of the Operation class, and not the name of the class itself.
- The name of the plugin (in this case, `special`).

As before, if your project uses `setup.cfg` or `setup.py` instead of `pyproject.toml`, see the [setuptools documentation](https://setuptools.pypa.io/en/latest/userguide/entry_point.html) for how to adapt these lines for your situation.

To check that your plugin is successfully detected by Qiskit, install your plugin package and follow the instructions at [Transpiler plugins](transpiler-plugins#list-available-high-level-synthesis-plugins) for listing installed plugins, and ensure that your plugin appears in the list:

In [7]:
from qiskit.transpiler.passes.synthesis import (
    high_level_synthesis_plugin_names,
)

high_level_synthesis_plugin_names("clifford")

['ag', 'bm', 'default', 'greedy', 'layers', 'lnn', 'rb_default']

Kung naka-install ang aming halimbawang plugin, lilitaw ang pangalang `my_unitary_synthesis` sa listahang ito.

Para ma-accommodate ang mga unitary synthesis plugin na naglalantad ng maraming opsyon,
ang plugin interface ay may opsyon para sa mga user na magbigay ng free-form
na configuration dictionary. Ito ay ipapasa sa `run` method
sa pamamagitan ng `options` keyword argument. Kung ang iyong plugin ay may mga configuration option na ito, dapat mong malinaw na idokumento ang mga ito.

## Halimbawa: Gumawa ng high-level synthesis plugin

Sa halimbawang ito, gagawa tayo ng high-level synthesis plugin na gumagamit lang ng built-in na function na [synth_clifford_bm](https://docs.quantum.ibm.com/api/qiskit/synthesis#synth_clifford_bm) para mag-synthesize ng Clifford operator.

Ang [HighLevelSynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin) class ay tumutukoy sa interface at kontrata para sa high-level synthesis plugins. Ang pangunahing method ay [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin#run).
Ang positional argument na `high_level_object` ay isang [Operation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Operation) na kumakatawan sa "high-level" na object na ise-synthesize. Halimbawa, maaari itong maging isang
[LinearFunction](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction) o isang
[Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford).
Ang mga sumusunod na keyword argument ay naroroon:
- Ang `target` ay tumutukoy sa target na backend, na nagbibigay-daan sa plugin
na ma-access ang lahat ng target-specific na impormasyon,
tulad ng coupling map, ang sinusuportahang gate set, at iba pa
- Ang `coupling_map` ay tumutukoy lamang sa coupling map, at ginagamit lamang kapag hindi tinukoy ang `target`.
- Ang `qubits` ay tumutukoy sa listahan ng mga Qubit kung saan
tinukoy ang high-level na object, sa kaso na ang synthesis ay ginagawa sa physical circuit.
Ang halagang ``None`` ay nagpapahiwatig na hindi pa napili ang layout at ang mga physical na Qubit sa target o coupling map na pinagtatrabahuhan ng operasyong ito ay hindi pa natutukoy.
- Ang `options`, isang free-form configuration dictionary para sa mga opsyon na tukoy sa plugin. Kung ang iyong plugin ay may mga configuration option na ito,
dapat mong malinaw na idokumento ang mga ito.

Ang `run` method ay nagbabalik ng [QuantumCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit)
na kumakatawan sa circuit na na-synthesize mula sa high-level na object na iyon.
Pinapayagan din na magbalik ng `None`, na nagpapahiwatig na hindi makakapag-synthesize ang plugin ng ibinigay na high-level na object.
Ang aktwal na synthesis ng mga high-level na object ay isinasagawa ng
[HighLevelSynthesis](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.HighLevelSynthesis)
transpiler pass.

Bukod sa `run` method, may ilang property method na kailangang tukuyin.
Tingnan ang [HighLevelSynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin) para sa dokumentasyon ng lahat ng kinakailangang property.

Tukuyin natin ang ating HighLevelSynthesisPlugin subclass: